In [1]:
"""
train_tender_model.py
=====================
Навчання моделі прогнозування ризику тендера ДО оголошення.

Особливості:
    - Тренуємо на ~12 млн тендерах зі СЛАБКОЮ міткою
      (0 = ДАСУ не моніторила АБО моніторила без порушень).
    - Тестуємо на тендерах з ДОСТОВІРНОЮ міткою — лише ті де
      ДАСУ реально проводила моніторинг (мітка точна).
    - Time-aware split: train < 2024-07, val 2024-07..2024-12, test 2025+.

Чому гібридна оцінка:
    На повному датасеті 99.5% — нулі (слабка мітка).
    Багато з цих нулів насправді приховують реальні порушення
    яких ДАСУ просто не виявила. Тому метрики на цих даних
    необ'єктивні. На тендерах з реальним моніторингом — мітка
    достовірна і метрики чесні.

Вхід:
    features_tenders_full.parquet
    dasu_labels.parquet — щоб знати які тендери реально перевірялися

Вихід:
    artifacts/tender_model_lgb.pkl
    artifacts/tender_calibrator.pkl
    artifacts/tender_model_meta.json
    artifacts/tender_shap_importance.csv
    artifacts/tender_curves.png
    artifacts/tender_risk_top.parquet  — топ-N найризикованіших тендерів
"""

import json
import warnings
import joblib
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap
import optuna
import lightgbm as lgb

from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import (
    average_precision_score, roc_auc_score,
    f1_score, precision_score, recall_score,
    precision_recall_curve, roc_curve,
    classification_report, confusion_matrix,
)

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ══════════════════════════════════════════════════════════════
# НАЛАШТУВАННЯ
# ══════════════════════════════════════════════════════════════

FEATURES_PATH    = "features_tenders_full.parquet"
DASU_LABELS_PATH = "dasu_labels.parquet"
ARTIFACTS_DIR    = Path("artifacts")
ARTIFACTS_DIR.mkdir(exist_ok=True)

RANDOM_STATE     = 42
N_OPTUNA_TRIALS  = 40
SAMPLE_FOR_HPO   = 1_000_000

TRAIN_CUTOFF     = pd.Timestamp("2024-07-01")
VAL_CUTOFF       = pd.Timestamp("2025-01-01")

NON_FEATURE_COLS = [
    "tender_id", "buyer_id", "tender_start_date",
    "dasu_label",
    "main_cpv_2_digit",
]

RISK_THRESHOLDS = {
    "Низький":    (0.00, 0.30),
    "Середній":   (0.30, 0.60),
    "Високий":    (0.60, 0.80),
    "Критичний":  (0.80, 1.01),
}


print("═" * 60)
print("1. ЗАВАНТАЖЕННЯ ДАНИХ")
print("═" * 60)

df = pd.read_parquet(FEATURES_PATH)
df["tender_start_date"] = pd.to_datetime(df["tender_start_date"])
print(f"Тендерів усього:    {len(df):,}")
print(f"Часовий діапазон:   {df['tender_start_date'].min().date()} → "
      f"{df['tender_start_date'].max().date()}")

vc = df["dasu_label"].value_counts()
print(f"\nРозподіл мітки:")
print(f"  label=0: {vc.get(0,0):,} ({vc.get(0,0)/len(df):.2%})")
print(f"  label=1: {vc.get(1,0):,} ({vc.get(1,0)/len(df):.2%})")

dasu_all = pd.read_parquet(DASU_LABELS_PATH, columns=["tender_id"])
certified_ids = set(dasu_all["tender_id"].astype(str).str.strip().tolist())
print(f"\nТендерів з реальним моніторингом ДАСУ: {len(certified_ids):,}")


print("\n" + "═" * 60)
print("2. ЧАСОВЕ РОЗБИТТЯ TRAIN / VAL / TEST")
print("═" * 60)

feature_cols = [c for c in df.columns if c not in NON_FEATURE_COLS]
print(f"Ознак для моделі: {len(feature_cols)}")
print(f"  {feature_cols}")

mask_train = df["tender_start_date"] < TRAIN_CUTOFF
mask_val   = (df["tender_start_date"] >= TRAIN_CUTOFF) & (df["tender_start_date"] < VAL_CUTOFF)
mask_test  = df["tender_start_date"] >= VAL_CUTOFF

mask_test_certified = mask_test & df["tender_id"].isin(certified_ids)

X_train = df.loc[mask_train, feature_cols].copy()
y_train = df.loc[mask_train, "dasu_label"].copy()

X_val   = df.loc[mask_val,   feature_cols].copy()
y_val   = df.loc[mask_val,   "dasu_label"].copy()

X_test_full = df.loc[mask_test,           feature_cols].copy()
y_test_full = df.loc[mask_test,           "dasu_label"].copy()
X_test_cert = df.loc[mask_test_certified, feature_cols].copy()
y_test_cert = df.loc[mask_test_certified, "dasu_label"].copy()

medians = X_train.median(numeric_only=True)
X_train = X_train.fillna(medians)
X_val   = X_val.fillna(medians)
X_test_full = X_test_full.fillna(medians)
X_test_cert = X_test_cert.fillna(medians)

print(f"\nTrain (слабкі мітки):       {len(X_train):,} | pos: {y_train.sum():,} ({y_train.mean():.3%})")
print(f"Val (слабкі мітки):         {len(X_val):,} | pos: {y_val.sum():,} ({y_val.mean():.3%})")
print(f"Test повний (слабкі мітки): {len(X_test_full):,} | pos: {y_test_full.sum():,} ({y_test_full.mean():.3%})")
print(f"Test достовірний (моніторовані): {len(X_test_cert):,} | pos: {y_test_cert.sum():,} ({y_test_cert.mean():.3%})")


print("\n" + "═" * 60)
print(f"3. OPTUNA HPO ({N_OPTUNA_TRIALS} спроб) на підвибірці {SAMPLE_FOR_HPO:,}")
print("═" * 60)

spw = (y_train == 0).sum() / max((y_train == 1).sum(), 1)
print(f"scale_pos_weight: {spw:.1f}")


def stratified_sample(X, y, n):
    if len(X) <= n:
        return X, y
    pos_idx = y[y == 1].index
    neg_idx = y[y == 0].index
    n_pos = min(len(pos_idx), int(n * y.mean()) + 1)
    n_neg = n - n_pos
    samp_idx = (
        pd.Index(np.random.RandomState(RANDOM_STATE).choice(neg_idx, n_neg, replace=False))
        .append(pd.Index(np.random.RandomState(RANDOM_STATE).choice(pos_idx, n_pos, replace=False)))
    )
    return X.loc[samp_idx], y.loc[samp_idx]

X_hpo, y_hpo = stratified_sample(X_train, y_train, SAMPLE_FOR_HPO)
print(f"Підвибірка для HPO: {len(X_hpo):,} | pos: {y_hpo.sum():,}")


def objective(trial: optuna.Trial) -> float:
    params = {
        "objective":         "binary",
        "metric":            "average_precision",
        "verbosity":         -1,
        "boosting_type":     "gbdt",
        "random_state":      RANDOM_STATE,
        "scale_pos_weight":  spw,
        "n_jobs":            -1,
        "num_leaves":        trial.suggest_int("num_leaves", 31, 127),
        "max_depth":         trial.suggest_int("max_depth", 4, 10),
        "learning_rate":     trial.suggest_float("learning_rate", 0.02, 0.15, log=True),
        "n_estimators":      trial.suggest_int("n_estimators", 300, 800),
        "min_child_samples": trial.suggest_int("min_child_samples", 50, 500),
        "subsample":         trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree":  trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha":         trial.suggest_float("reg_alpha", 1e-3, 5.0, log=True),
        "reg_lambda":        trial.suggest_float("reg_lambda", 1e-3, 5.0, log=True),
    }
    m = lgb.LGBMClassifier(**params)
    m.fit(
        X_hpo, y_hpo,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(30, verbose=False),
                   lgb.log_evaluation(-1)],
    )
    prob = m.predict_proba(X_val)[:, 1]
    return average_precision_score(y_val, prob)


study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
)
study.optimize(objective, n_trials=N_OPTUNA_TRIALS, show_progress_bar=True)

print("BEST PARAMS:", study.best_params)
print("BEST VALUE:", study.best_value)

best_params = {
    **study.best_params,
    "objective": "binary", "metric": "average_precision",
    "verbosity": -1, "boosting_type": "gbdt",
    "random_state": RANDOM_STATE, "scale_pos_weight": spw,
    "n_jobs": -1,
}
print(f"\nНайкращий val PR-AUC: {study.best_value:.4f}")


print("\n" + "═" * 60)
print("4. ФІНАЛЬНЕ НАВЧАННЯ НА ПОВНОМУ TRAIN")
print("═" * 60)

n_calib = int(len(X_train) * 0.20)
X_sub, X_calib   = X_train.iloc[:-n_calib], X_train.iloc[-n_calib:]
y_sub, y_calib   = y_train.iloc[:-n_calib], y_train.iloc[-n_calib:]

lgb_model = lgb.LGBMClassifier(**best_params)
lgb_model.fit(
    X_sub, y_sub,
    eval_set=[(X_val, y_val)],
    callbacks=[lgb.early_stopping(50, verbose=False),
               lgb.log_evaluation(100)],
)

val_raw = lgb_model.predict_proba(X_val)[:, 1]
print(f"\n[Val] PR-AUC (без калібрування): {average_precision_score(y_val, val_raw):.4f}")
print(f"[Val] ROC-AUC:                   {roc_auc_score(y_val, val_raw):.4f}")


print("\n" + "═" * 60)
print("5. КАЛІБРУВАННЯ (ізотонічна регресія)")
print("═" * 60)

calib_raw = lgb_model.predict_proba(X_calib)[:, 1]
calibrator = IsotonicRegression(out_of_bounds="clip")
calibrator.fit(calib_raw, y_calib)

val_cal      = calibrator.predict(val_raw)
test_cert_raw = lgb_model.predict_proba(X_test_cert)[:, 1]
test_cert_cal = calibrator.predict(test_cert_raw)
test_full_raw = lgb_model.predict_proba(X_test_full)[:, 1]
test_full_cal = calibrator.predict(test_full_raw)

print(f"[Val] PR-AUC (калібрований): {average_precision_score(y_val, val_cal):.4f}")


print("\n" + "═" * 60)
print("6. ВИБІР ПОРОГУ КЛАСИФІКАЦІЇ")
print("═" * 60)

prec_v, rec_v, thr_v = precision_recall_curve(y_val, val_cal)
f1_v   = 2 * prec_v[:-1] * rec_v[:-1] / (prec_v[:-1] + rec_v[:-1] + 1e-9)
best_i = np.argmax(f1_v)
best_threshold = float(thr_v[best_i])

print(f"Оптимальний поріг (max F1 на val): {best_threshold:.4f}")
print(f"  Precision: {prec_v[best_i]:.4f}")
print(f"  Recall:    {rec_v[best_i]:.4f}")
print(f"  F1:        {f1_v[best_i]:.4f}")


print("\n" + "═" * 60)
print("7. ФІНАЛЬНІ МЕТРИКИ")
print("═" * 60)

def report(y_true, y_prob, name):
    pred = (y_prob >= best_threshold).astype(int)
    print(f"\n── {name} ──")
    print(f"  Розмір:      {len(y_true):,}")
    print(f"  Позитивних:  {y_true.sum():,} ({y_true.mean():.2%})")
    print(f"  PR-AUC:      {average_precision_score(y_true, y_prob):.4f}")
    print(f"  ROC-AUC:     {roc_auc_score(y_true, y_prob):.4f}")
    if y_true.sum() > 0:
        print(f"  F1:          {f1_score(y_true, pred):.4f}")
        print(f"  Precision:   {precision_score(y_true, pred):.4f}")
        print(f"  Recall:      {recall_score(y_true, pred):.4f}")
        print(f"  Confusion matrix:\n{confusion_matrix(y_true, pred)}")

report(y_test_cert, test_cert_cal, "TEST ДОСТОВІРНИЙ (тільки моніторовані ДАСУ)")
report(y_test_full, test_full_cal, "TEST ПОВНИЙ (зі слабкими мітками)")


print("\n" + "═" * 60)
print("8. SHAP — ВАЖЛИВІСТЬ ОЗНАК")
print("═" * 60)

explainer = shap.TreeExplainer(lgb_model)
shap_sample = X_test_cert.sample(min(2000, len(X_test_cert)), random_state=RANDOM_STATE)
sv = explainer.shap_values(shap_sample)
sv = sv[1] if isinstance(sv, list) else sv

importance = (
    pd.DataFrame({
        "feature": feature_cols,
        "mean_abs_shap": np.abs(sv).mean(axis=0),
    })
    .sort_values("mean_abs_shap", ascending=False)
)
print(importance.to_string(index=False))
importance.to_csv(ARTIFACTS_DIR / "tender_shap_importance.csv", index=False)

fig, ax = plt.subplots(figsize=(9, 6), facecolor='white')
ax.set_facecolor('white')

top = importance.head(15)
ax.barh(top["feature"][::-1], top["mean_abs_shap"][::-1], color="#5B9BD5")
ax.set_xlabel("Середній |SHAP|")
ax.set_title("Важливість ознак тендерної моделі")
plt.tight_layout()

plt.savefig(ARTIFACTS_DIR / "tender_shap_bar.png", dpi=150, facecolor='white', transparent=False, bbox_inches='tight')
plt.close()


fig, axes = plt.subplots(1, 2, figsize=(13, 5))

pt, rt, _ = precision_recall_curve(y_test_cert, test_cert_cal)
axes[0].plot(rt, pt, color="#E24B4A", lw=2,
             label=f"Test certified (PR-AUC={average_precision_score(y_test_cert, test_cert_cal):.3f})")
axes[0].axhline(y_test_cert.mean(), ls="--", color="gray", lw=1, label="Випадкова")
axes[0].set(xlabel="Повнота", ylabel="Точність",
            title="PR-крива (test certified)", xlim=[0,1], ylim=[0,1])
axes[0].legend()

fp, tp, _ = roc_curve(y_test_cert, test_cert_cal)
axes[1].plot(fp, tp, color="#5B9BD5", lw=2,
             label=f"AUC={roc_auc_score(y_test_cert, test_cert_cal):.3f}")
axes[1].plot([0,1],[0,1], ls="--", color="gray", lw=1)
axes[1].set(xlabel="FPR", ylabel="TPR",
            title="ROC-крива (test certified)", xlim=[0,1], ylim=[0,1])
axes[1].legend()

plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / "tender_curves.png", dpi=150)
plt.close()


print("\n" + "═" * 60)
print("10. ТОП-1000 НАЙРИЗИКОВАНІШИХ ТЕНДЕРІВ ТЕСТУ")
print("═" * 60)

df_test_info = df.loc[mask_test, ["tender_id", "buyer_id", "tender_start_date"]].copy()
df_test_info["risk_prob"]   = test_full_cal
df_test_info["risk_score"]  = (test_full_cal * 100).round(1)
df_test_info["was_flagged"] = (test_full_cal >= best_threshold).astype("int8")
df_test_info["dasu_label"]  = y_test_full.values

top_risky = df_test_info.nlargest(1000, "risk_prob").reset_index(drop=True)
top_risky.to_parquet(ARTIFACTS_DIR / "tender_risk_top.parquet",
                     compression="snappy", engine="pyarrow")
print(f"Збережено топ-1000 у tender_risk_top.parquet")
print(top_risky.head(10).to_string(index=False))


joblib.dump(lgb_model,   ARTIFACTS_DIR / "tender_model_lgb.pkl")
joblib.dump(calibrator,  ARTIFACTS_DIR / "tender_calibrator.pkl")
joblib.dump(explainer,   ARTIFACTS_DIR / "tender_shap_explainer.pkl")
joblib.dump(medians,     ARTIFACTS_DIR / "tender_feature_medians.pkl")

meta = {
    "best_threshold":   best_threshold,
    "feature_cols":     feature_cols,
    "risk_thresholds":  RISK_THRESHOLDS,
    "scale_pos_weight": float(spw),
    "best_params":      study.best_params,
    "temporal_split": {
        "train_end": str(TRAIN_CUTOFF.date()),
        "val_end":   str(VAL_CUTOFF.date()),
    },
    "test_certified_metrics": {
        "pr_auc":    float(average_precision_score(y_test_cert, test_cert_cal)),
        "roc_auc":   float(roc_auc_score(y_test_cert, test_cert_cal)),
    },
}
with open(ARTIFACTS_DIR / "tender_model_meta.json", "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

print(f"\n✅ Артефакти збережено у {ARTIFACTS_DIR}/")

C:\Users\legen\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


════════════════════════════════════════════════════════════
1. ЗАВАНТАЖЕННЯ ДАНИХ
════════════════════════════════════════════════════════════
Тендерів усього:    922,575
Часовий діапазон:   2021-09-01 → 2025-12-28

Розподіл мітки:
  label=0: 908,147 (98.44%)
  label=1: 14,428 (1.56%)

Тендерів з реальним моніторингом ДАСУ: 78,132

════════════════════════════════════════════════════════════
2. ЧАСОВЕ РОЗБИТТЯ TRAIN / VAL / TEST
════════════════════════════════════════════════════════════
Ознак для моделі: 19
  ['proc_method_enc', 'non_price_criteria', 'is_works', 'is_services', 'is_high_risk_cpv', 'log_tender_value', 'submission_window_days', 'has_submission_window', 'number_of_items', 'number_of_documents', 'is_buyer_masked', 'is_weekend', 'is_q4', 'is_december', 'buyer_violation_rate', 'buyer_total_tenders', 'value_vs_cpv_median', 'window_vs_cpv_median', 'value_vs_buyer_median']

Train (слабкі мітки):       448,565 | pos: 9,636 (2.148%)
Val (слабкі мітки):         181,691 | pos: 1,

Best trial: 39. Best value: 0.278635: 100%|██████████| 40/40 [02:57<00:00,  4.45s/it]


BEST PARAMS: {'num_leaves': 88, 'max_depth': 7, 'learning_rate': 0.035940390680637944, 'n_estimators': 556, 'min_child_samples': 117, 'subsample': 0.9084388799738065, 'colsample_bytree': 0.9339759861586499, 'reg_alpha': 2.956880057689818, 'reg_lambda': 1.3919031438357383}
BEST VALUE: 0.27863516593160725

Найкращий val PR-AUC: 0.2786

════════════════════════════════════════════════════════════
4. ФІНАЛЬНЕ НАВЧАННЯ НА ПОВНОМУ TRAIN
════════════════════════════════════════════════════════════
[100]	valid_0's average_precision: 0.261648
[200]	valid_0's average_precision: 0.274067

[Val] PR-AUC (без калібрування): 0.2741
[Val] ROC-AUC:                   0.9632

════════════════════════════════════════════════════════════
5. КАЛІБРУВАННЯ (ізотонічна регресія)
════════════════════════════════════════════════════════════
[Val] PR-AUC (калібрований): 0.2579

════════════════════════════════════════════════════════════
6. ВИБІР ПОРОГУ КЛАСИФІКАЦІЇ
═══════════════════════════════════════════════

In [2]:
import matplotlib.pyplot as plt

# 1. Примусово скидаємо стиль (іноді допомагає саме це)
plt.style.use('default')

fig, ax = plt.subplots(figsize=(9, 6), facecolor='white')
ax.set_facecolor('white')

top = importance.head(15)
ax.barh(top["feature"][::-1], top["mean_abs_shap"][::-1], color="#5B9BD5")

# 2. ЯВНО задаємо чорний колір для тексту та підписів
ax.set_xlabel("Середній |SHAP|", color='black')
ax.set_title("Важливість ознак тендерної моделі", color='black')
ax.tick_params(axis='x', colors='black')
ax.tick_params(axis='y', colors='black')

# 3. Робимо самі рамки графіка (осі) чорними
for spine in ax.spines.values():
    spine.set_edgecolor('black')

plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / "tender_shap_bar.png", dpi=150, facecolor='white', transparent=False, bbox_inches='tight')
plt.close()

## ОЦЕ ВАЖЛИВО - ТУТ ГЛЯЖУ ШО ПО РОЗПОДІЛУ ЙМОВІРНОСТЕЙ ВЖЕ ПІСЛЯ КАЛІБРАЦІЇ

In [ ]:
# Розподіл каліброваних значень
df = pd.read_parquet("features_tenders_full.parquet")
df["tender_start_date"] = pd.to_datetime(df["tender_start_date"])
test = df[df["tender_start_date"] >= "2025-01-01"]

# Завантажуємо модель
model = joblib.load("artifacts/tender_model_lgb.pkl")
medians = joblib.load("artifacts/tender_feature_medians.pkl")

with open("artifacts/tender_model_meta.json", encoding="utf-8") as f:
    import json
    meta = json.load(f)

feature_cols = meta["feature_cols"]
X = test[feature_cols].fillna(medians)

raw  = model.predict_proba(X)[:, 1]
cal  = calibrator.predict(raw)

# Розподіл унікальних каліброваних значень
print("\n\nРозподіл каліброваних ймовірностей у тесті 2025:")
print(f"Унікальних значень: {len(np.unique(cal))}")

stat = pd.DataFrame({
    "calibrated_prob": cal,
    "label": test["dasu_label"].values,
})

# Згрупуємо за каліброваним значенням
summary = stat.groupby("calibrated_prob").agg(
    n_tenders=("label", "count"),
    n_violations=("label", "sum"),
).reset_index()
summary["actual_violation_rate"] = summary["n_violations"] / summary["n_tenders"]
summary = summary.sort_values("calibrated_prob", ascending=False)

print(summary.to_string(index=False))

## ГРАФІК ДЛЯ КУРСАЧА - РОЗПОДІЛ РИЗИКУ ВРАХОВУЮЧИ КАЛІБРУВАННЯ (буковками шоб писало а не циферками)

In [ ]:
import pandas as pd
import joblib, json
from pathlib import Path

# Завантажуємо ризики для всього test
df = pd.read_parquet("features_tenders_full.parquet")
df["tender_start_date"] = pd.to_datetime(df["tender_start_date"])
test = df[df["tender_start_date"] >= "2025-01-01"]

model = joblib.load("artifacts/tender_model_lgb.pkl")
calibrator = joblib.load("artifacts/tender_calibrator.pkl")
medians = joblib.load("artifacts/tender_feature_medians.pkl")
meta = json.loads(Path("artifacts/tender_model_meta.json").read_text(encoding="utf-8"))

X = test[meta["feature_cols"]].fillna(medians)
raw = model.predict_proba(X)[:, 1]
cal = calibrator.predict(raw)

# Нові межі риск-біннінгу
THRESHOLDS = [
    ("Низький",    0.00, 0.05),
    ("Помірний",   0.05, 0.15),
    ("Високий",    0.15, 0.30),
    ("Критичний",  0.30, 1.01),
]

print(f"{'Рівень':>12} {'тендерів':>12} {'порушень':>12} {'фактич.':>10} {'очікув.':>10}")
print("─" * 60)
for name, lo, hi in THRESHOLDS:
    mask = (cal >= lo) & (cal < hi)
    n = mask.sum()
    viol = test["dasu_label"].values[mask].sum()
    rate = viol / n if n > 0 else 0
    expected = cal[mask].mean() if n > 0 else 0
    print(f"{name:>12} {n:>12,} {viol:>12,} {rate:>10.2%} {expected:>10.2%}")